# 3LC setup
See the [3LC Quickstart](https://docs.3lc.ai/3lc/latest/quickstart/quickstart.html) for more details on the sections below.

## Create a free 3LC account

Before getting started, make sure you have a free 3LC account. Go to https://3lc.ai and click "Sign Up" in the upper right corner. Sign up for a 3LC account (or log in to an existing one) using your preferred method. Then copy your API Key from the 3LC account home page.

## Create a Python environment

Create a Python environment called "3LC" using your preferred tool:
- `python -m venv 3LC`
  - See the [Python venv documentation](https://docs.python.org/3.12/library/venv.html) for details
- `conda create -n 3LC`
  - See the [conda documentation](https://docs.conda.io/projects/conda/en/stable/user-guide/getting-started.html) for details, or reach out if you are having trouble

## Install the 3LC YOLO integration

With your "3LC" Python environment activated, install the 3LC YOLO integration with the following command. This will also install the `3lc` Python package and all required dependencies.

```bash
pip install git+https://github.com/3lc-ai/3lc-ultralytics@develop
```

## Configure 3LC with your API key

Configure 3LC to use the API key you copied earlier (or get it [here](https://account.3lc.ai/api-key)).

```bash
3lc login <paste API key here>
```

## Start the 3LC Object Service

The [3LC Object Service](https://docs.3lc.ai/3lc/latest/user-guide/object-service/index.html#object-service) is responsible for serving your dataset and metrics to the 3LC Dashboard. It is started from the terminal and can be terminated by pressing Q.

```bash
3lc service --no-public-examples
```

## Launch the 3LC Dashboard

After starting the Object Service, launch the 3LC Dashboard in a browser at https://dashboard.3lc.ai and log in to your 3LC account. Your 3LC project will be displayed in the Dashboard once you begin creating 3LC data below. See the [3LC Dashboard](https://docs.3lc.ai/3lc/latest/user-guide/dashboard/index.html#dashboard-index) documentation for more details.

Note that if you want to browse example 3LC projects (in addition to your own), you can start the Object Service in the step above without specifying the `--no-public-examples` argument.

# Verify 3LC setup

Now we are almost ready to start creating 3LC `Table`s for your dataset, and `Run`s for your training runs with Ultralytics YOLO.

We first import 3LC to verify that the installation and configuration was successful. Make sure you have done the `3lc login` step above to avoid prompts or errors related to specifying a 3LC API key.

In [ ]:
import tlc

# Creating 3LC Tables

When you generate new images and labels in Duality, you can run the following code to create 3LC Tables for that data.

In [ ]:
import tlc
PROJECT_NAME = "Duality-3LC-Kaggle"  # Place all 3LC Tables and Runs in the same project

# This for loop allows you to create multiple 3LC Tables (e.g., train and val sets) in one go
for split in ["train", "val"]:
    table = tlc.Table.from_yolo(
        dataset_yaml_file="path/to/yaml/file.yaml",  # the yolo_params.yaml file in the data folder you generate from Falcon
        split=split,
        table_name="initial",
        dataset_name=split,
        project_name=PROJECT_NAME,
    )

    print(f"Created table with URL: {table.url}")

# Joining 3LC Tables

When you generate more data and would like to combine it with an existing 3LC Table, first create a Table with the previous snippet, then join it with your existing Table using the following code.

In [ ]:
import tlc
PROJECT_NAME = "Duality-3LC-Kaggle"  # Place all 3LC Tables and Runs in the same project

table_1 = tlc.Table.from_url("path/to/table_1/url")  # Hint: Copy Table URLs from Dashboard
table_2 = tlc.Table.from_url("path/to/table_2/url")

joined_table = tlc.Table.join_tables(
    tables=[table_1, table_2],
    table_name="initial",
    dataset_name="joined_train",
    project_name=PROJECT_NAME,
    description="Join table_1 and table_2",
)

# Training a YOLO model

The following code trains a YOLO model with the 3LC YOLO integration.

In [ ]:
from tlc_ultralytics import Settings, YOLO
import tlc
PROJECT_NAME = "Duality-3LC-Kaggle"  # Place all 3LC Tables and Runs in the same project

RUN_NAME = "run-1"  # Define the run name to organize all your runs in a nice way

# Set 3LC specific settings
settings = Settings(
    project_name=PROJECT_NAME,
    run_name=RUN_NAME,
    run_description="description of the run",
)

# Update the URLs for the train and val tables when you make data revisions in 3LC Dashboard
train_table = tlc.Table.from_url("/path/to/train/table/url")  # Hint: Copy Table URLs from Dashboard
val_table = tlc.Table.from_url("/path/to/val/table/url")

model = YOLO("yolov8s.pt")

# You may add any YOLO arguments here
model.train(
    tables={"train": train_table, "val": val_table},
    settings=settings,
    epochs=5,
    agnostic_nms=True,
    project=PROJECT_NAME,
    name=RUN_NAME,
)

After training, the models (`best.pt` and `last.pt`) will be saved at `../Duality-3LC-Kaggle/<RUN_NAME>/weights/`.

# Analyze training runs in the 3LC Dashboard

- Watch this [video](https://youtu.be/ZDjBmuRhh2U)
- More details in [How-to tutorials](https://docs.3lc.ai/3lc/latest/how-to/index.html)
